### Import important stuff

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download(
    "SPY",
    period = "60d",
    interval = "15m" 
)

spy.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Datetime,,,,,
2026-07-14 09:30:00-04:00,751.309998,751.520020,749.440002,750.909973,4275781
2026-07-14 09:45:00-04:00,751.309998,751.789978,750.580017,751.304993,1408483
2026-07-14 10:00:00-04:00,750.429993,751.479980,749.779907,751.320007,1227309
2026-07-14 10:15:00-04:00,750.710022,751.299988,748.659973,750.419983,1322623
2026-07-14 10:30:00-04:00,751.094971,753.309998,750.479980,750.710022,1858157


In [2]:
spy.shape

(1552, 5)

In [3]:
spy.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1552 entries, 2026-07-14 09:30:00-04:00 to 2026-10-06 13:45:00-04:00
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, SPY)   1552 non-null   float64
 1   (High, SPY)    1552 non-null   float64
 2   (Low, SPY)     1552 non-null   float64
 3   (Open, SPY)    1552 non-null   float64
 4   (Volume, SPY)  1552 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 72.8 KB


In [4]:
spy['Close'].describe()

Ticker,SPY
count,1552.000000
mean,762.108472
std,11.358948
min,729.539978
25%,756.855011
50%,765.419952
75%,770.395020
max,781.340027


In [5]:
spy['log_returns'] = np.log(spy['Close']/spy['Close'].shift())
spy[['Close', 'log_returns']].head()

Price,Close,log_returns
Ticker,SPY,
Datetime,,
2026-07-14 09:30:00-04:00,751.309998,NaN
2026-07-14 09:45:00-04:00,751.309998,0.000000
2026-07-14 10:00:00-04:00,750.429993,-0.001172
2026-07-14 10:15:00-04:00,750.710022,0.000373
2026-07-14 10:30:00-04:00,751.094971,0.000513


In [6]:
spy.columns = [
    col[0] if col[1] == '' else col[0]
    for col in spy.columns
]
print(spy.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'log_returns'], dtype='str')


In [7]:
spy['log_returns'].describe()

count    1551.000000
mean        0.000024
std         0.001332
min        -0.013038
25%        -0.000506
50%         0.000020
75%         0.000506
max         0.009952
Name: log_returns, dtype: float64

In [8]:
# Sharpe
spy['log_returns'].mean()/spy['log_returns'].std()

np.float64(0.018258233472679485)

### We create lags and see their correlations

In [9]:
spy['lag_1'] = spy['log_returns'].shift()
spy['lag_2'] = spy['log_returns'].shift(2)
spy['lag_3'] = spy['log_returns'].shift(3)

spy[['log_returns', 'lag_1', 'lag_2', 'lag_3']]

,log_returns,lag_1,lag_2,lag_3
Datetime,,,,
2026-07-14 09:30:00-04:00,NaN,NaN,NaN,NaN
2026-07-14 09:45:00-04:00,0.000000,NaN,NaN,NaN
2026-07-14 10:00:00-04:00,-0.001172,0.000000,NaN,NaN
2026-07-14 10:15:00-04:00,0.000373,-0.001172,0.000000,NaN
2026-07-14 10:30:00-04:00,0.000513,0.000373,-0.001172,0.000000
...,...,...,...,...
2026-10-06 12:45:00-04:00,-0.000465,0.000108,-0.000922,-0.000102
2026-10-06 13:00:00-04:00,-0.000540,-0.000465,0.000108,-0.000922
2026-10-06 13:15:00-04:00,0.000128,-0.000540,-0.000465,0.000108


In [10]:
spy[['log_returns', 'lag_1', 'lag_2', 'lag_3']].dropna().corr()

,log_returns,lag_1,lag_2,lag_3
log_returns,1.000000,0.020213,-0.035487,-0.016894
lag_1,0.020213,1.000000,0.020028,-0.035503
lag_2,-0.035487,0.020028,1.000000,0.020024
lag_3,-0.016894,-0.035503,0.020024,1.000000


In [11]:
spy = spy.dropna(subset=[
    'log_returns',
    'lag_1',
    'lag_2',
    'lag_3'
])
spy

,Close,High,Low,Open,Volume,log_returns,lag_1,lag_2,lag_3
Datetime,,,,,,,,,
2026-07-14 10:30:00-04:00,751.094971,753.309998,750.479980,750.710022,1858157,0.000513,0.000373,-0.001172,0.000000
2026-07-14 10:45:00-04:00,751.710022,751.960022,750.469971,751.109924,788674,0.000819,0.000513,0.000373,-0.001172
2026-07-14 11:00:00-04:00,752.549988,753.070007,751.280029,751.710022,1493455,0.001117,0.000819,0.000513,0.000373
2026-07-14 11:15:00-04:00,752.659973,752.710022,751.650024,752.590027,715442,0.000146,0.001117,0.000819,0.000513
2026-07-14 11:30:00-04:00,751.559998,752.900024,750.760010,752.640015,1479819,-0.001463,0.000146,0.001117,0.000819
...,...,...,...,...,...,...,...,...,...
2026-10-06 12:45:00-04:00,780.021423,780.469971,779.859985,780.400024,828247,-0.000465,0.000108,-0.000922,-0.000102
2026-10-06 13:00:00-04:00,779.600098,780.099976,779.500000,780.020081,725847,-0.000540,-0.000465,0.000108,-0.000922
2026-10-06 13:15:00-04:00,779.700012,779.760010,778.900024,779.609985,946107,0.000128,-0.000540,-0.000465,0.000108


### We start building the magnitude model(Second Order Auto Regression Model(AR(2)))

In [12]:
y = spy['log_returns'].values
y

array([0.00051265, 0.00081854, 0.00111678, ..., 0.00012815, 0.00025649,
       0.00035892], shape=(1548,))

In [13]:
X = spy[['lag_1', 'lag_2', 'lag_3']].values
X

array([[ 0.00037309, -0.00117198,  0.        ],
       [ 0.00051265,  0.00037309, -0.00117198],
       [ 0.00081854,  0.00051265,  0.00037309],
       ...,
       [-0.00054029, -0.00046461,  0.00010755],
       [ 0.00012815, -0.00054029, -0.00046461],
       [ 0.00025649,  0.00012815, -0.00054029]], shape=(1548, 3))

In [14]:
#Import torch stuff
import torch
import torch.nn as nn
import torch.optim as optim

In [15]:
#Train Model
def train_model(model, criterion, optimizer, X_train, y_train, X_test, y_test, no_epochs=5000,verbose=False):
    for epoch in range(no_epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred,y_train)
        loss.backward()
        optimizer.step()

    if epoch % 500 == 0:
        print("Epoch: ", epoch, "Loss: ", loss.item())

    #print("Final weight: ", model.weight.data)
    #print("Final Bias: ", model.bias.data)


In [16]:
# Retrieve ML model's prediction for out-sample(test)
def test_model_predictions(model, df_test, X_test, target):
    model.eval()

    with torch.no_grad():
        y_hat = model(X_test).flatten()

    y_actual = torch.tensor(
        df_test[target].values,
        dtype=torch.float32
    ).flatten()

    # MAE
    mae = torch.mean(torch.abs(y_actual - y_hat)).item()

    # MSE
    mse = torch.mean((y_actual - y_hat) ** 2).item()

    # RMSE
    rmse = torch.sqrt(torch.tensor(mse)).item()

    # R^2
    ss_res = torch.sum((y_actual - y_hat) ** 2)
    ss_tot = torch.sum((y_actual - torch.mean(y_actual)) ** 2)

    r2 = (1 - ss_res / ss_tot).item()

    #For now, I only show MAE

    return {
    "MAE": mae,
    "MSE": mse,
    "R2": r2,
    "model": model
    }
    

In [17]:
#Evaluate Accuracy of the model
def evaluate_model_predictions(df_train, df_test, features, target):
    no_features = len(features)
    model = nn.Linear(no_features, 1)
    criterion = nn.HuberLoss()
    optimizer = optim.SGD(model.parameters(),lr=0.01)

    X_train, X_test = df_train[features], df_test[features]
    y_train, y_test = df_train[target], df_test[target]

    X_train = torch.tensor(X_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test.values, dtype=torch.float32)

    y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

    train_model(model, criterion, optimizer, X_train, y_train, X_test, y_test, no_epochs=5000,verbose=False)
    return test_model_predictions(model, df_test, X_test, target)


In [18]:
spy['abs_log_returns'] = spy['log_returns'].abs()
spy['abs_lag_1'] = spy['lag_1'].abs()
spy['abs_lag_2'] = spy['lag_2'].abs()
spy['abs_lag_3'] = spy['lag_3'].abs()

features = ['abs_lag_1', 'abs_lag_2']
target = 'abs_log_returns'

### Back Testing

### Cross Validation: Time Series Splits

In [19]:
#Time split code
def timesplit(df,train_size=0.75):
    i = int(len(df) * train_size)
    return df[:i].copy(), df[i:].copy()

In [20]:
# Show results for different CV train/test splits
mean_maes = []
mean_mses = []
mean_r_twos = []
models = []
for num_legs in [1,2,3]:
    target = 'abs_log_returns'
    features = []
    for feature_num in range(1,num_legs+1):
        features.append(f"abs_lag_{feature_num}")

    #train_sizes = [] # Not important for now
    maes = []
    mses = []
    r_twos = []
    # weights = [] # Not important for now
    # biases = [] # Not important for now

    for train_split in [0.4 + 0.1 * i for i in range(6)]:

        spy_train, spy_test = timesplit(spy, train_size=train_split)

        result = evaluate_model_predictions(
            spy_train,
            spy_test,
            features,
            target
        )

        model = result["model"]

        #train_sizes.append(train_split) # Not important for now
        maes.append(result["MAE"])
        mses.append(result["MSE"])
        r_twos.append(result["R2"])
        # weights.append(model.weight.detach().numpy().flatten()) # Not important for now
        # biases.append(model.bias.item()) # Not important for now
    print(f"AR({num_legs})\n")
    print("MAES:", maes, "\n")
    print("MSES:", mses, "\n")
    print("R2:", r_twos, "\n")
    models.append(f"AR({num_legs})")
    mean_maes.append(np.mean(maes))
    mean_mses.append(np.mean(mses))
    mean_r_twos.append(np.mean(r_twos))

cv_results = pd.DataFrame({
    'Model': models,
    'Mean MAE': mean_maes,
    'Mean MSE': mean_mses,
    #'Mean R2': mean_r_twos # Useless for magnitudes
})

cv_results

AR(1)

MAES: [0.0010845452779904008, 0.0009334103669971228, 0.0006034011603333056, 0.0006850272184237838, 0.0006938016158528626, 0.0006379325059242547] 

MSES: [2.186553729188745e-06, 1.8810341089192661e-06, 1.1730824098776793e-06, 1.7318091067863861e-06, 1.1608398153839516e-06, 1.4757015378563665e-06] 

R2: [-1.2803442478179932, -0.8099050521850586, -0.045255184173583984, -0.37523651123046875, -0.16975247859954834, -0.4143674373626709] 

AR(2)

MAES: [0.0006582547212019563, 0.0007002929924055934, 0.0006382380961440504, 0.0009360680123791099, 0.0007740678265690804, 0.0005773607408627868] 

MSES: [1.1174327028129483e-06, 1.2068926480424125e-06, 1.2143544836362707e-06, 1.999510459427256e-06, 1.3204603419580963e-06, 1.1783769195972127e-06] 

R2: [-0.16536414623260498, -0.16125547885894775, -0.08202993869781494, -0.5878192186355591, -0.33059847354888916, -0.129400372505188] 

AR(3)

MAES: [0.0007596277864649892, 0.0006848051561973989, 0.0007471545832231641, 0.0007153867627494037, 0.0005981

,Model,Mean MAE,Mean MSE
0,AR(1),0.000773,0.000002
1,AR(2),0.000714,0.000001
2,AR(3),0.000683,0.000001


### Cross Validation: Rolling Window

In [21]:
# Show results for different CV Rlling Window
mean_maes = []
mean_mses = []
mean_r_twos = []
models = []
for num_legs in [1,2,3]:
    target = 'abs_log_returns'
    features = []
    for feature_num in range(1,num_legs+1):
        features.append(f"abs_lag_{feature_num}")

    #window_no = [] # Not important for now
    maes = []
    mses = []
    r_twos = []
    # weights = [] # Not important for now
    # biases = [] # Not important for now
    window_size = 130

    for i in range(6):

        train_start, train_end = window_size * i, window_size * (i+1)
        test_start, test_end = window_size * (i+1), window_size * (i+2)

        spy_train = spy[train_start:train_end].copy()
        spy_test = spy[test_start: test_end].copy()

        result = evaluate_model_predictions(
            spy_train,
            spy_test,
            features,
            target
        )

        model = result["model"]

        #window_no.append(train_split) # Not important for now
        maes.append(result["MAE"])
        mses.append(result["MSE"])
        r_twos.append(result["R2"])
        # weights.append(model.weight.detach().numpy().flatten()) # Not important for now
        # biases.append(model.bias.item()) # Not important for now
    print(f"AR({num_legs})\n")
    print("MAES:", maes, "\n")
    print("MSES:", mses, "\n")
    print("R2:", r_twos, "\n")
    models.append(f"AR({num_legs})")
    mean_maes.append(np.mean(maes))
    mean_mses.append(np.mean(mses))
    mean_r_twos.append(np.mean(r_twos))

cv_results = pd.DataFrame({
    'Model': models,
    'Mean MAE': mean_maes,
    'Mean MSE': mean_mses,
    #'Mean R2': mean_r_twos # Useless for magnitudes
})

cv_results

AR(1)

MAES: [0.0007485965033993125, 0.0009266411070711911, 0.0011272246483713388, 0.00042346646660007536, 0.0005448812735266984, 0.0007115714834071696] 

MSES: [1.212265487993136e-06, 2.2283998077909928e-06, 1.915965412990772e-06, 6.117154498497257e-07, 7.090569056344975e-07, 1.341420215794642e-06] 

R2: [-0.03132665157318115, -0.11702942848205566, -1.5362532138824463, -0.3760155439376831, -0.3482295274734497, -0.799105167388916] 

AR(2)

MAES: [0.0007996316417120397, 0.0008622169261798263, 0.0006533916457556188, 0.0004910060088150203, 0.00046652223682031035, 0.0004906891845166683] 

MSES: [1.3030743275521672e-06, 1.8843503539756057e-06, 8.978335017673089e-07, 4.980922199138149e-07, 5.506742581928847e-07, 7.91969966940087e-07] 

R2: [-0.10858166217803955, 0.05543208122253418, -0.18850433826446533, -0.12042725086212158, -0.047074317932128906, -0.06218564510345459] 

AR(3)

MAES: [0.0007372079417109489, 0.0013416135916486382, 0.0008611934608779848, 0.00044938313658349216, 0.000567842507

,Model,Mean MAE,Mean MSE
0,AR(1),0.000747,1.336471e-06
1,AR(2),0.000627,9.876658e-07
2,AR(3),0.000762,1.526393e-06


In [22]:
# Show results for different CV Expanding Window
mean_maes = []
mean_mses = []
mean_r_twos = []
models = []
for num_legs in [1,2,3]:
    target = 'abs_log_returns'
    features = []
    for feature_num in range(1,num_legs+1):
        features.append(f"abs_lag_{feature_num}")

    #window_no = [] # Not important for now
    maes = []
    mses = []
    r_twos = []
    # weights = [] # Not important for now
    # biases = [] # Not important for now
    window_size = 130

    for i in range(6):

        train_start = 0
        train_end = window_size + i * window_size
        test_start = train_end
        test_end = test_start + window_size

        spy_train = spy[train_start:train_end].copy()
        spy_test = spy[test_start: test_end].copy()

        result = evaluate_model_predictions(
            spy_train,
            spy_test,
            features,
            target
        )

        model = result["model"]

        #window_no.append(train_split) # Not important for now
        maes.append(result["MAE"])
        mses.append(result["MSE"])
        r_twos.append(result["R2"])
        # weights.append(model.weight.detach().numpy().flatten()) # Not important for now
        # biases.append(model.bias.item()) # Not important for now
    print(f"AR({num_legs})\n")
    print("MAES:", maes, "\n")
    print("MSES:", mses, "\n")
    print("R2:", r_twos, "\n")
    models.append(f"AR({num_legs})")
    mean_maes.append(np.mean(maes))
    mean_mses.append(np.mean(mses))
    mean_r_twos.append(np.mean(r_twos))

cv_results = pd.DataFrame({
    'Model': models,
    'Mean MAE': mean_maes,
    'Mean MSE': mean_mses,
    #'Mean R2': mean_r_twos # Useless for magnitudes
})

cv_results

AR(1)

MAES: [0.0006865470786578953, 0.0014550182968378067, 0.0006591323181055486, 0.0005567944026552141, 0.0005681041511707008, 0.0005334151792339981] 

MSES: [1.1304144891255419e-06, 4.960546448273817e-06, 8.455552347186313e-07, 5.659195494445157e-07, 6.587824259440822e-07, 8.026627256185748e-07] 

R2: [0.0383075475692749, -1.486572027206421, -0.11930108070373535, -0.2730005979537964, -0.2526355981826782, -0.07652664184570312] 

AR(2)

MAES: [0.0009452379890717566, 0.001531038200482726, 0.0009817277314141393, 0.0010489342967048287, 0.000649387133307755, 0.0009417927940376103] 

MSES: [1.6425343574155704e-06, 5.396508186095161e-06, 1.586008806953032e-06, 1.323186779700336e-06, 7.627256763953483e-07, 1.3879555353923934e-06] 

R2: [-0.3973749876022339, -1.7051067352294922, -1.0994741916656494, -1.9764256477355957, -0.4502774477005005, -0.861518144607544] 

AR(3)

MAES: [0.0008465130813419819, 0.0008723409264348447, 0.00072196259861812, 0.0011710088001564145, 0.0007758820429444313, 0.000

,Model,Mean MAE,Mean MSE
0,AR(1),0.000743,0.000001
1,AR(2),0.001016,0.000002
2,AR(3),0.000851,0.000001


### Still yet to decide on the best model from data above

### Classification model to predict sign of log_returns

In [23]:
# Creating Binary classification target
spy['log_return_dir'] = (spy['log_returns'] > 0).astype(int)
#check target balance
spy['log_return_dir'].value_counts()

log_return_dir
1    787
0    761
Name: count, dtype: int64

In [34]:
#Test Model Accuracy
def test_model_accuracy(model, df_test, X_test, target):
    model.eval()

    with torch.no_grad():
        logits = model(X_test).flatten()

        # Convert logits to probabilities
        probabilities = torch.sigmoid(logits)

        # Convert probabilities to 0/1 predictions
        y_hat = (probabilities >= 0.5).float()

    # Actual classes
    y_actual = torch.tensor(
        df_test[target].values,
        dtype=torch.float32
    ).flatten()

    # Accuracy
    accuracy = (y_hat == y_actual).float().mean().item()

    return {
        "Accuracy": accuracy,
        "model": model
    }

In [ ]:
#Train model
#Evaluate Accuracy of the model
def evaluate_cmodel_predictions(df_train, df_test, features, target):
    no_features = len(features)
    model = nn.Linear(no_features, 1)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(),lr=0.01)

    X_train, X_test = df_train[features], df_test[features]
    y_train, y_test = df_train[target], df_test[target]

    X_train = torch.tensor(X_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test.values, dtype=torch.float32)

    y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

    
    train_model(model, criterion, optimizer, X_train, y_train, X_test, y_test, no_epochs=5000,verbose=False)
    return test_model_accuracy(model, df_test, X_test, target)

In [ ]:
# Time split checks
models = []
mean_accuracies = []
for num_legs in [1,2,3]:
    print(f"AR({num_legs})")
    target = 'log_return_dir'
    features = []
    for feature_num in range(1,num_legs+1):
        features.append(f"lag_{feature_num}")

    #train_sizes = [] # Not important for now
    # weights = [] # Not important for now
    # biases = [] # Not important for now
    accuracies = []
    for train_split in [0.4 + 0.1 * i for i in range(6)]:
        #print(f"\nClassification balance for {train_split:.0%} train / {(1-train_split):.0%} test")
        spy_train, spy_test = timesplit(spy, train_size=train_split)
        #print("Train:")
        #print(spy_train['log_return_dir'].value_counts())
        
        #print("Test:")
        #print(spy_test['log_return_dir'].value_counts())

        result = evaluate_cmodel_predictions(
            spy_train,
            spy_test,
            features,
            target
        )

        model = result["model"]
        accuracies.append(result["Accuracy"])

        #train_sizes.append(train_split) # Not important for now
        # weights.append(model.weight.detach().numpy().flatten()) # Not important for now
        # biases.append(model.bias.item()) # Not important for now
    print(accuracies)
    models.append(f"AR({num_legs})")
    mean_accuracies.append(np.mean(accuracies))

cv_results = pd.DataFrame({
    'Model': models,
    'Mean Accuracies': mean_accuracies,
})

cv_results

AR(1)
[0.5026910901069641, 0.4767441749572754, 0.4951612949371338, 0.46451613306999207, 0.42580646276474, 0.5548387169837952]
AR(2)
[0.5026910901069641, 0.4767441749572754, 0.5, 0.46451613306999207, 0.42580646276474, 0.5548387169837952]
AR(3)
[0.5026910901069641, 0.4767441749572754, 0.5145161151885986, 0.46451613306999207, 0.42580646276474, 0.5548387169837952]


,Model,Mean Accuracies
0,AR(1),0.486626
1,AR(2),0.487433
2,AR(3),0.489852


### Conclusion

For Classification model, Third order autoregressive model when trained on 60% and tested on 40% data was the best.

For Magnitude model,First order Autoregressive model wen trained on Rolling Window for i = 3(fourth iteration) is the best.